In [1]:
# Check GPU Runtime
!nvidia-smi

Thu Aug  1 22:35:35 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.104.05             Driver Version: 535.104.05   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla T4                       Off | 00000000:00:04.0 Off |                    0 |
| N/A   71C    P8              12W /  70W |      0MiB / 15360MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [2]:
!pip install -q -U bitsandbytes --use-deprecated=legacy-resolver
!pip install -q -U git+https://github.com/huggingface/transformers.git --use-deprecated=legacy-resolver
!pip install -q -U git+https://github.com/huggingface/peft.git --use-deprecated=legacy-resolver
!pip install -q -U git+https://github.com/huggingface/accelerate.git --use-deprecated=legacy-resolver
!pip install -q -U datasets scipy ipywidgets matplotlib --use-deprecated=legacy-resolver

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 MB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.7/19.7 MB 26.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.8/547.8 kB 14.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.1/41.1 MB 23.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.4/139.4 kB 12.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 115.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.9/39.9 MB 21.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━

In [3]:
!pip install boto3
!pip install fastapi uvicorn[standard] nest-asyncio pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.2/139.2 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 62.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.7/82.7 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.2/92.2 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 341.4/341.4 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.9/71.9 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 87.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 1

In [ ]:
import os
import boto3
from botocore.exceptions import NoCredentialsError

aws_access_key_id = os.environ['AWS_ACCESS_KEY_ID']  # AWS Access Key ID
aws_secret_access_key = os.environ['AWS_SECRET_ACCESS_KEY'] # AWS Secret Access Key
aws_region = 'us-east-1'

# Create a session using your AWS credentials
session = boto3.Session(
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    region_name=aws_region
)

In [5]:
# Define your S3 bucket name and folder (if any)
bucket_name = 'archistackoverflow'
s3_folder = 'checkpoint-500'  # if files are in the root, set to ''

# Create an S3 client
s3 = session.client('s3')

# List files in the specified S3 folder
response = s3.list_objects_v2(Bucket=bucket_name, Prefix=s3_folder)

if 'Contents' in response:
    files = [content['Key'] for content in response['Contents']]
    print("Files in S3 bucket:")
    for file in files:
        print(file)
else:
    print("No files found in the S3 bucket or folder.")

Files in S3 bucket:
checkpoint-500/README.md
checkpoint-500/adapter_config.json
checkpoint-500/adapter_model.safetensors
checkpoint-500/optimizer.pt
checkpoint-500/rng_state.pth
checkpoint-500/scheduler.pt
checkpoint-500/trainer_state.json
checkpoint-500/training_args.bin


In [6]:
import os

# Create a local directory to store downloaded files
local_dir = '/content/s3_downloads'
os.makedirs(local_dir, exist_ok=True)

# Download each file from S3
for file_key in files:
    local_file_path = os.path.join(local_dir, os.path.basename(file_key))
    try:
        s3.download_file(bucket_name, file_key, local_file_path)
        print(f"Downloaded {file_key} to {local_file_path}")
    except NoCredentialsError:
        print("Credentials not available")
    except FileNotFoundError:
        print(f"The file {file_key} was not found in S3")

Downloaded checkpoint-500/README.md to /content/s3_downloads/README.md
Downloaded checkpoint-500/adapter_config.json to /content/s3_downloads/adapter_config.json
Downloaded checkpoint-500/adapter_model.safetensors to /content/s3_downloads/adapter_model.safetensors
Downloaded checkpoint-500/optimizer.pt to /content/s3_downloads/optimizer.pt
Downloaded checkpoint-500/rng_state.pth to /content/s3_downloads/rng_state.pth
Downloaded checkpoint-500/scheduler.pt to /content/s3_downloads/scheduler.pt
Downloaded checkpoint-500/trainer_state.json to /content/s3_downloads/trainer_state.json
Downloaded checkpoint-500/training_args.bin to /content/s3_downloads/training_args.bin


In [ ]:
from huggingface_hub import login
login(token=os.environ['HF_TOKEN'])
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
!pip install pyngrok

In [ ]:
##### FOR WEBSOCKET CONNECTION
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from fastapi import FastAPI, Request, WebSocket, WebSocketDisconnect
from fastapi.responses import JSONResponse
import nest_asyncio
from pyngrok import ngrok
import uvicorn
from fastapi.middleware.cors import CORSMiddleware

nest_asyncio.apply()
# Initialize FastAPI
app = FastAPI()
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # Or specify your domain
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Initialize the model and tokenizer
base_model_id = "mistralai/Mistral-7B-v0.1"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

eval_tokenizer = AutoTokenizer.from_pretrained(base_model_id, add_bos_token=True, trust_remote_code=True)

from peft import PeftModel

ft_model = PeftModel.from_pretrained(base_model, "/content/s3_downloads")
ft_model.eval()

@app.get("/")
def read_root():
    return {"message": "Hello World"}

@app.post("/generate/")
async def generate_text(request: Request):
    data = await request.json()
    prompt = data.get("prompt")
    model_input = eval_tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        output = ft_model.generate(**model_input, max_new_tokens=200, repetition_penalty=1.15)
    response = eval_tokenizer.decode(output[0], skip_special_tokens=True)
    return JSONResponse(content={"generated_text": response})

# Allow FastAPI to run within Colab's event loop
# nest_asyncio.apply()
ngrok.set_auth_token(os.environ['NGROK_AUTH_TOKEN'])

# Optional: Set a custom subdomain (requires a paid plan)
public_url = ngrok.connect(8000, bind_tls=True)
print(f"Public URL: {public_url}")

# Start the FastAPI server
uvicorn.run(app, host="0.0.0.0", port=8000)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse
import nest_asyncio
from pyngrok import ngrok
import uvicorn
from fastapi.middleware.cors import CORSMiddleware

# Apply nest_asyncio to allow FastAPI to run in the Colab environment
nest_asyncio.apply()

# Initialize FastAPI
app = FastAPI()
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # Or specify your domain
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Initialize the model and tokenizer
base_model_id = "mistralai/Mistral-7B-v0.1"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

eval_tokenizer = AutoTokenizer.from_pretrained(base_model_id, add_bos_token=True, trust_remote_code=True)

from peft import PeftModel

ft_model = PeftModel.from_pretrained(base_model, "/content/s3_downloads")
ft_model.eval()

@app.get("/")
def read_root():
    return {"message": "Hello World"}

@app.post("/generate/")
async def generate_text(request: Request):
    data = await request.json()
    prompt = data.get("prompt")
    model_input = eval_tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        output = ft_model.generate(**model_input, max_new_tokens=200, repetition_penalty=1.15)
    response = eval_tokenizer.decode(output[0], skip_special_tokens=True)
    return JSONResponse(content={"generated_text": response})

# Start ngrok tunnel
ngrok.set_auth_token(os.environ['NGROK_AUTH_TOKEN'])
public_url = ngrok.connect(8000)
print(f"Public URL: {public_url}")

# Start the FastAPI server
uvicorn.run(app, host="0.0.0.0", port=8000)